# Retranslate 900k VQA pairs + Build RAG Index

End-to-end pipeline on a single A100:

**Part 1 — Retranslate** the full 900k English VQA dataset to Romanian using
vLLM with continuous batching. One `llm.generate` call per chunk maximises
A100 throughput; the loop is resume-safe and checkpoints to Drive.

**Part 2 — Build RAG.** Embed the retranslated questions with a multilingual
sentence-transformer, embed the unique images with CLIP, and persist both
collections in a ChromaDB folder.

**Why vLLM:** continuous batching on A100 reaches hundreds of examples/second,
making 900k translations finish in a few hours rather than days.

---
# Part 1 — Retranslation

## 1.1 Install

In [ ]:
%%capture
!pip install -q vllm ijson

## 1.2 Mount Drive + config

In [ ]:
import os
try:
    from google.colab import drive
    if not os.path.exists('/content/drive/MyDrive'):
        drive.mount('/content/drive')
    else:
        print('Drive already mounted.')
except ImportError:
    print('Not running in Colab.')

In [ ]:
# Translation model (recommended on A100: a 14B-32B in bf16)
MODEL_NAME = "unsloth/gemma-4-31B-it"
# Faster alternatives: 'Qwen/Qwen2.5-14B-Instruct', 'Qwen/Qwen2.5-7B-Instruct'

# Source dataset (English fields under *_eng) and output path
IN_FILE  = "/content/drive/My Drive/vqa_clean/dataset_final/train_ro_plus_eng.json"
OUT_FILE = "/content/drive/My Drive/vqa_clean/dataset_final/train_900k_retranslated_v4.json"

# Examples per Drive checkpoint
CHECKPOINT_EVERY = 100_000
MAX_TOKENS = 128

## 1.3 Load dataset + resume from previous checkpoint

In [ ]:
import json

print("Loading source dataset...")
with open(IN_FILE, "r", encoding="utf-8") as f:
    data = json.load(f)
print(f"Total source examples: {len(data):,}")

if os.path.exists(OUT_FILE):
    print("Found previous progress, resuming...")
    with open(OUT_FILE, "r", encoding="utf-8") as f:
        saved = json.load(f)
    for k, v in saved.items():
        if v.get("question_retranslated"):
            data[k] = v
    del saved

pending = [k for k, v in data.items() if not v.get("question_retranslated")]
print(f"Already translated: {len(data) - len(pending):,} | Remaining: {len(pending):,}")

## 1.4 Start vLLM

In [ ]:
import sys
os.environ["VLLM_USE_FLASHINFER_SAMPLER"] = "0"

# Wrap stdout/stderr so vLLM's fileno() calls do not fail inside notebooks.
class _FD:
    def __init__(self, f, fd):
        self._f, self._fd = f, fd
    def __getattr__(self, n):
        return getattr(self._f, n)
    def fileno(self):
        return self._fd
    def flush(self):
        try: self._f.flush()
        except Exception: pass

sys.stdout = _FD(sys.stdout, 1)
sys.stderr = _FD(sys.stderr, 2)

from vllm import LLM, SamplingParams
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
llm = LLM(
    model = MODEL_NAME,
    dtype = "bfloat16",
    max_model_len = 2048,
    gpu_memory_utilization = 0.90,
    trust_remote_code = True,
)

## 1.5 Sampling + prompt builder

In [ ]:
sampling_params = SamplingParams(temperature=0.1, max_tokens=MAX_TOKENS)

SYS_PROMPT = (
    "You are an expert English to Romanian translator specialising in Visual "
    "Question Answering (VQA). Translate the 'Question' and 'Answer' to "
    "natural, grammatical Romanian. Return only valid JSON with keys "
    "'question_ro' and 'answer_ro'."
)

def build_prompt(item):
    q_en = item.get("question_eng", "")
    a_en = item.get("answer_eng", "")
    user = (
        f"Question: {q_en}\n"
        f"Answer: {a_en}\n\n"
        'Return ONLY: {"question_ro": "...", "answer_ro": "..."}'
    )
    messages = [
        {"role": "system", "content": SYS_PROMPT},
        {"role": "user",   "content": user},
    ]
    return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

## 1.6 Translate — one `llm.generate` per chunk

We give vLLM the entire remaining batch in one call; it handles continuous
batching internally. Chunks of `CHECKPOINT_EVERY` examples are checkpointed
to Drive so a disconnected session can resume.

In [ ]:
import time
import shutil
import re

def save_checkpoint():
    local_tmp = "/content/_checkpoint.json"
    with open(local_tmp, "w", encoding="utf-8") as f:
        json.dump(data, f, ensure_ascii=False)
        f.flush()
        os.fsync(f.fileno())
    shutil.copy(local_tmp, OUT_FILE)

start = time.time()
total_done = 0

for chunk_start in range(0, len(pending), CHECKPOINT_EVERY):
    chunk_ids = pending[chunk_start:chunk_start + CHECKPOINT_EVERY]
    prompts   = [build_prompt(data[k]) for k in chunk_ids]
    outputs   = llm.generate(prompts, sampling_params)

    for k, out in zip(chunk_ids, outputs):
        txt = out.outputs[0].text.strip().replace("```json", "").replace("```", "")
        try:
            parsed = json.loads(txt)
            data[k]["question_retranslated"] = parsed.get("question_ro", "")
            data[k]["answer_retranslated"]   = parsed.get("answer_ro", "")
        except Exception:
            # Fallback regex parse for slightly malformed JSON
            m_q = re.search(r'"question_ro"\s*:\s*"([^"]*)"', txt)
            m_a = re.search(r'"answer_ro"\s*:\s*"([^"]*)"',   txt)
            data[k]["question_retranslated"] = m_q.group(1) if m_q else ""
            data[k]["answer_retranslated"]   = m_a.group(1) if m_a else ""

    total_done += len(chunk_ids)
    save_checkpoint()
    elapsed = time.time() - start
    rate    = total_done / max(elapsed, 1e-9)
    eta_min = (len(pending) - total_done) / max(rate, 1e-9) / 60
    print(f"Checkpoint: {total_done:,}/{len(pending):,} "
          f"| {rate:.1f} ex/s | ETA {eta_min:.0f} min")

print("Translation done.")

## 1.7 Quality check on a sample

In [ ]:
import random
translated = [v for v in data.values() if v.get("question_retranslated")]
empty      = sum(1 for v in data.values() if v.get("question_retranslated") == "")
print(f"Translated: {len(translated):,} | Empty: {empty:,}\n")
for v in random.sample(translated, min(8, len(translated))):
    print(f"EN: {v.get('question_eng','')!r} -> {v.get('answer_eng','')!r}")
    print(f"RO: {v['question_retranslated']!r} -> {v['answer_retranslated']!r}\n")

## 1.8 Free GPU before RAG

In [ ]:
import gc, torch, contextlib
try:
    from vllm.distributed.parallel_state import destroy_model_parallel
    destroy_model_parallel()
except Exception:
    pass
del llm
gc.collect()
with contextlib.suppress(Exception):
    torch.cuda.empty_cache()
!nvidia-smi --query-gpu=memory.used,memory.total --format=csv

---
# Part 2 — Build the RAG index

In [ ]:
%%capture
!pip install -q chromadb sentence-transformers pillow tqdm

## 2.1 Config

In [ ]:
RAG_DB_DIR  = "/content/drive/My Drive/vqa_clean/rag_db_900k"
IMAGES_ZIP  = "/content/drive/My Drive/licenta/images.zip"
RETRANSLATED_JSON = OUT_FILE
os.makedirs(RAG_DB_DIR, exist_ok=True)

## 2.2 Unzip images locally

In [ ]:
import glob
os.makedirs("/content/data", exist_ok=True)
!unzip -q -o "$IMAGES_ZIP" -d /content/data

IMAGE_DIR = None
for c in ["/content/data/images", "/content/data"]:
    if os.path.isdir(c) and glob.glob(os.path.join(c, "*.jpg")):
        IMAGE_DIR = c
        break
if IMAGE_DIR is None:
    any_jpg = glob.glob("/content/data/**/*.jpg", recursive=True)
    IMAGE_DIR = os.path.dirname(any_jpg[0])
print(f"IMAGE_DIR={IMAGE_DIR} | images={len(glob.glob(os.path.join(IMAGE_DIR, '*.jpg')))}")

## 2.3 Build the record list

In [ ]:
if "data" not in globals():
    with open(RETRANSLATED_JSON, encoding="utf-8") as f:
        data = json.load(f)

records = []
for qid, v in data.items():
    q = v.get("question_retranslated") or v.get("question", "")
    a = v.get("answer_retranslated")   or v.get("answer_clean") or v.get("answer", "")
    if not q:
        continue
    records.append({
        "questionId":    qid,
        "imageId":       str(v.get("imageId", "")),
        "question":      q,
        "answer":        a,
        "fullAnswer":    v.get("fullAnswer", "") or "",
        "caption_llava": v.get("caption_llava", "") or "",
    })
print(f"Records to index: {len(records):,}")

## 2.4 Load embedding models

In [ ]:
!pip uninstall -y torchcodec
os.environ["TRANSFORMERS_NO_TORCHCODEC"] = "1"

import torch
from sentence_transformers import SentenceTransformer

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
TEXT_MODEL_NAME  = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
IMAGE_MODEL_NAME = "clip-ViT-B-32"

text_model  = SentenceTransformer(TEXT_MODEL_NAME,  device=DEVICE)
image_model = SentenceTransformer(IMAGE_MODEL_NAME, device=DEVICE)
print(f"Embeddings on {DEVICE}")

## 2.5 Create ChromaDB collections

In [ ]:
import chromadb
client = chromadb.PersistentClient(path=RAG_DB_DIR)
for name in ["questions", "images"]:
    try:
        client.delete_collection(name)
    except Exception:
        pass
q_collection   = client.create_collection("questions", metadata={"hnsw:space": "cosine"})
img_collection = client.create_collection("images",    metadata={"hnsw:space": "cosine"})
print("Collections created.")

## 2.6 Index questions

In [ ]:
from tqdm.auto import tqdm

BATCH = 1024
for s in tqdm(range(0, len(records), BATCH), desc="Indexing questions"):
    b   = records[s:s + BATCH]
    qs  = [r["question"] for r in b]
    embs = text_model.encode(qs, batch_size=BATCH, normalize_embeddings=True, show_progress_bar=False)
    q_collection.add(
        ids        = [r["questionId"] for r in b],
        embeddings = embs.tolist(),
        metadatas  = [{
            "questionId":    r["questionId"],
            "imageId":       r["imageId"],
            "question":      r["question"],
            "answer":        r["answer"],
            "fullAnswer":    r["fullAnswer"],
            "caption_llava": r["caption_llava"],
        } for r in b],
        documents  = qs,
    )
print(f"Questions indexed: {q_collection.count():,}")

## 2.7 Index unique images

In [ ]:
from PIL import Image

seen = {}
for r in records:
    seen.setdefault(r["imageId"], r)
unique_ids = list(seen.keys())
print(f"Unique images: {len(unique_ids):,}")

def load_img(image_id):
    p = os.path.join(IMAGE_DIR, image_id + ".jpg")
    if not os.path.exists(p):
        return None
    try:
        return Image.open(p).convert("RGB")
    except Exception:
        return None

IMG_BATCH = 128
missing = []
buf_ids, buf_imgs = [], []

def flush():
    if not buf_imgs:
        return
    embs = image_model.encode(buf_imgs, batch_size=IMG_BATCH,
                              normalize_embeddings=True, show_progress_bar=False)
    img_collection.add(
        ids        = list(buf_ids),
        embeddings = embs.tolist(),
        metadatas  = [{
            "imageId":  iid,
            "question": seen[iid]["question"],
            "answer":   seen[iid]["answer"],
        } for iid in buf_ids],
        documents  = list(buf_ids),
    )
    buf_ids.clear()
    buf_imgs.clear()

for iid in tqdm(unique_ids, desc="Embedding images"):
    img = load_img(iid)
    if img is None:
        missing.append(iid)
        continue
    buf_ids.append(iid)
    buf_imgs.append(img)
    if len(buf_imgs) >= IMG_BATCH:
        flush()
flush()
print(f"Images indexed: {img_collection.count():,} | Missing: {len(missing):,}")

## 2.8 Save manifest

In [ ]:
import datetime

manifest = {
    "created":          datetime.datetime.now().isoformat(),
    "source_json":      RETRANSLATED_JSON,
    "text_model":       TEXT_MODEL_NAME,
    "image_model":      IMAGE_MODEL_NAME,
    "text_dim":         text_model.get_sentence_embedding_dimension(),
    "image_dim":        image_model.get_sentence_embedding_dimension(),
    "n_questions":      q_collection.count(),
    "n_images":         img_collection.count(),
    "n_missing_images": len(missing),
    "distance":         "cosine",
}
with open(os.path.join(RAG_DB_DIR, "manifest.json"), "w", encoding="utf-8") as f:
    json.dump(manifest, f, ensure_ascii=False, indent=2)
print("RAG index ready.")
print(json.dumps(manifest, ensure_ascii=False, indent=2))